In [1]:
import pandas as pd
import numpy as np

# 1. Configuration: Dates and Hours
start_date = '2022-01-01'
end_date = '2025-12-31'
all_days = pd.date_range(start=start_date, end=end_date, freq='D')
business_hours = [f"{h:02d}:00" for h in range(8, 17)] # 08:00 to 16:00 (9 hours)

# 2. Create the Master Timeline
master_timeline = pd.DataFrame([
    {'Date': d.strftime('%Y-%m-%d'), 'Time': h}
    for d in all_days for h in business_hours
])
master_timeline['Date_dt'] = pd.to_datetime(master_timeline['Date'])

# 3. Load and Prepare Festival Data
df_festivals = pd.read_csv('../data/aberdeen_festivals_expanded.csv')
df_festivals['Date'] = pd.to_datetime(df_festivals['Date'])

# 4. Load and Prepare Match Data
df_matches = pd.read_csv('../data/aberdeen_home_matches_importance.csv')
df_matches['Date'] = pd.to_datetime(df_matches['Date'])

# 5. Merge Festivals into Timeline
final_events = master_timeline.merge(
    df_festivals, left_on='Date_dt', right_on='Date', how='left', suffixes=('', '_drop')
)
final_events = final_events.rename(columns={
    'Event Name': 'festival_name',
    'Importance': 'festival_importance'
})
final_events['is_festival'] = final_events['festival_name'].notna().astype(int)
final_events['festival_importance'] = final_events['festival_importance'].fillna(0)
final_events['festival_name'] = final_events['festival_name'].fillna('None')

# 6. Merge Matches into Timeline
final_events = final_events.merge(
    df_matches, left_on='Date_dt', right_on='Date', how='left', suffixes=('', '_match_drop')
)
final_events = final_events.rename(columns={
    'Opponent': 'match_opponent',
    'Importance': 'match_importance'
})
final_events['is_match_day'] = final_events['match_opponent'].notna().astype(int)
final_events['match_importance'] = final_events['match_importance'].fillna(0)
final_events['match_opponent'] = final_events['match_opponent'].fillna('None')

# 7. Finalize Structure
# Adding the combined 'Date' column for easier merging with weather/sales
final_events.insert(0, 'Date_full', pd.to_datetime(final_events['Date'] + ' ' + final_events['Time']))

# Keep only the columns needed for the model
cols_to_keep = [
    'Date_full',
    'is_festival', 'festival_importance', 'festival_name',
    'is_match_day', 'match_importance', 'match_opponent'
]
final_events = final_events[cols_to_keep].rename(columns={'Date_full': 'Date'})

# 8. Save the single combined file
final_events.to_csv('aberdeen_events_master_timeline.csv', index=False)

print(f"Unified events timeline created with {len(final_events)} rows.")
print(final_events.head())

Unified events timeline created with 13167 rows.
                 Date  is_festival  festival_importance festival_name  \
0 2022-01-01 08:00:00            0                  0.0          None   
1 2022-01-01 09:00:00            0                  0.0          None   
2 2022-01-01 10:00:00            0                  0.0          None   
3 2022-01-01 11:00:00            0                  0.0          None   
4 2022-01-01 12:00:00            0                  0.0          None   

   is_match_day  match_importance match_opponent  
0             0               0.0           None  
1             0               0.0           None  
2             0               0.0           None  
3             0               0.0           None  
4             0               0.0           None  


In [2]:
final_events.drop(columns=['festival_name', 'match_opponent'], errors='ignore', inplace=True)

In [3]:
final_events.to_csv('processed_csv/aberdeen_events_master_timeline.csv', index=False)


In [4]:
def check_missing_dates(df):
    """
    Checks if there are any missing hours (08:00-16:00) in the processed DataFrame.
    """
    if 'Date' not in df.columns:
        print("Error: 'Date' column not found in DataFrame.")
        return
    
    # Ensure Date is datetime
    df_dates = pd.to_datetime(df['Date'])
    
    start_dt = df_dates.min()
    end_dt = df_dates.max()
    
    if pd.isna(start_dt) or pd.isna(end_dt):
        print("Error: Could not determine start or end date.")
        return

    # Generate full range (08:00 to 16:00)
    full_range = pd.date_range(start=start_dt.normalize(), end=end_dt.normalize() + pd.Timedelta(days=1), freq='h')
    full_range = full_range[(full_range.hour >= 8) & (full_range.hour < 17)]
    
    missing = full_range.difference(df_dates)
    
    print(f"\n--- Final Data Check ---")
    print(f"Date Range: {start_dt} to {end_dt}")
    print(f"Expected Rows: {len(full_range)}")
    print(f"Actual Rows: {len(df_dates)}")
    
    if len(missing) > 0:
        print(f"WARNING: There are {len(missing)} missing timestamps!")
        if len(missing) > 10:
            print("First 10 missing:")
            print(missing[:10])
        else:
            print(missing)
    else:
        print("SUCCESS: No missing timestamps found in the 08:00-16:00 range.")

# Run the check
check_missing_dates(final_events)


--- Final Data Check ---
Date Range: 2022-01-01 08:00:00 to 2025-12-31 16:00:00
Expected Rows: 13149
Actual Rows: 13167
SUCCESS: No missing timestamps found in the 08:00-16:00 range.
